In [3]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications.vgg16 import VGG16
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.vgg16 import preprocess_input
import json
import pandas as pd
import numpy as np
import os

In [4]:
#mostly relied on googles AI overview for this cell. finding a non AI version in the meantime


model = VGG16(weights='imagenet', include_top=False, pooling='avg')

def get_embedding(img_path):
    img = image.load_img(img_path, target_size=(224, 224))
    x = image.img_to_array(img)
    x = np.expand_dims(x, axis=0)
    x = preprocess_input(x)
    return model.predict(x)


images = '/Users/aditibelle/Desktop/pubmed_set/images/'

image_files = [f for f in os.listdir(images) if f.lower().endswith(('.jpg', '.png', '.jpeg'))]

feature_embeddings = []


for filename in image_files:
        img_path = os.path.join(images, filename)

        embedding = get_embedding(img_path)

        feature_embeddings.append(embedding) 
print(feature_embeddings)

print(len(feature_embeddings))



1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 193ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 112ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 107ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 110ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 109ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 107ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 109ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 108ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 109ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 110ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 108ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 109ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 109ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 107ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 107ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 107ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 105ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 106ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 106ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 106ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 107ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 107ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 106ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 109ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 105ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



In [5]:
embeddingsfile = 'VGGembeddings.npy'
def save_embeddings(embeddings_list, embeddingsfile):
    """
    Saves embeddings as .npy file

    input: 
        embeddings_list: list of embeddings generated from model

    output:
        None

    """
    np.save(embeddingsfile, embeddings_list)

save_embeddings(feature_embeddings, embeddingsfile)

print("Saved embedding.npy")

Saved embedding.npy


In [6]:
print(np.array(feature_embeddings).dtype)

float32


In [114]:
temp = np.load(embeddingsfile)

In [115]:
temp.shape

(1, 1, 512)

In [116]:
temp.dtype

dtype('float32')

In [117]:
type (feature_embeddings)

list

In [118]:
np.array_equal(np.array(feature_embeddings), temp) 

True

In [119]:
#loading the images and their captions, will be used for image retrieval


# Define path for PubMed dataset
pubmed_path = '/Users/aditibelle/Desktop/pubmed_set/captions.json'

# Function to load JSON data into a DataFrame
def load_captions(json_path):
    with open(json_path, 'r') as file:
        data = json.load(file)
    # Convert to DataFrame and transpose for better readability
    df = pd.DataFrame(data).T
    return df

# Load the PubMed dataset
pubmed_df = load_captions(pubmed_path)

# Retrieve the captions and UUIDs from the DataFrame
pubmed_captions = pubmed_df['caption'].tolist()
pubmed_uuids = pubmed_df['uuid'].tolist()

# Display the first few entries
pubmed_df.head()

,caption,uuid
0,"ER expression in tumor tissue. IHC staining, o...",3f93c716-8fc9-42e9-bc29-bec52a51ab4b
1,Nuclear expression of TS (brown) in a colon ca...,9fcdf1e1-139c-4b63-bf1a-79d83c71f41a
2,Nuclear expression of E2F1 (brown) in a colon ...,00f1ad7a-f4b0-4938-b874-089d40a123ce
3,Cytoplasmic immunoexpression of PD-L1 in oral ...,9d3aef30-7c8b-4b78-9acf-ec523f952650
4,Nuclear and perinuclear immunoexpression of Fo...,b317d529-3626-49fc-9282-e4f28cf3d1cb


In [120]:
#image retrieval

import faiss
import torch

In [121]:
num_images = 3272
embedding_dim = 4096  
feature_embeddings = torch.rand(num_images, embedding_dim) 

In [122]:
# Convert embeddings to NumPy array
embeddings_np = feature_embeddings.numpy().astype('float32')

# Normalize embeddings (if using cosine similarity)
faiss.normalize_L2(embeddings_np)

# Build FAISS index
index = faiss.IndexFlatIP(embeddings_np.shape[1])  # Inner product for cosine similarity
index.add(embeddings_np)

print(f"FAISS index contains {index.ntotal} vectors.")

FAISS index contains 3272 vectors.


In [123]:
# After generating embeddings
# Convert embeddings to NumPy array
embeddings_np = feature_embeddings.numpy().astype('float32')


# Save the processed image paths and captions
np.save('processed_image_paths.npy', processed_image_paths)
np.save('processed_captions.npy', processed_captions)

NameError: name 'processed_image_paths' is not defined